# Stokastisk 0.1.0 — quickstart on Kaggle (2 × Tesla T4)

Settings: **Accelerator: GPU T4 × 2**, **Internet: ON**, Python 3.13. One line installs the SDK with the bundled vLLM 0.25.1, Triton attention and the CUDA 13.0 `sm_75` binaries (downloaded from the Hugging Face CDN, SHA256‑verified by pip). Nothing has to be uploaded as a dataset.

In [ ]:
# 1. install (one line)
!pip install stokastisk --extra-index-url https://ai-biruni.github.io/stokastisk/simple/

In [ ]:
# 2. check the installation (no GPU engine is started here)
import stokastisk, json
print('stokastisk', stokastisk.__version__)
print(json.dumps(stokastisk.prohibited_report(), indent=1))   # clean = no FlashAttention / FlashInfer / FlashMLA anywhere
print(stokastisk.check_model('Qwen/Qwen2.5-0.5B-Instruct'))

In [ ]:
%%writefile demo.py
# vLLM starts its engine in a separate process (spawn): keep the engine code in a script with a __main__ guard.
import stokastisk

if __name__ == '__main__':
    schema = {'type': 'object', 'properties': {'city': {'type': 'string', 'maxLength': 40}, 'population': {'type': 'integer', 'minimum': 0, 'maximum': 100000000}},
              'required': ['city', 'population'], 'additionalProperties': False}   # bound open-ended fields: a grammar cannot make a model stop
    with stokastisk.LLM('Qwen/Qwen2.5-0.5B-Instruct', config=stokastisk.StokastiskConfig(max_model_len=2048)) as llm:
        print('attention backend :', llm.verify_attention_backend())           # TRITON_ATTN
        print('completion        :', llm.complete('The capital of France is', max_tokens=16))
        print('chat              :', llm.chat([{'role': 'user', 'content': 'Say hello in five words.'}])[0].outputs[0].text)
        print('structured        :', llm.generate_structured(['Give a JSON object describing Paris.'], schema, max_tokens=96))

In [ ]:
!python demo.py

## More
* Two GPUs: `StokastiskConfig(tensor_parallel_size=2)` (a model too big for one T4) or `stokastisk.parallel.ReplicaPool(model, devices=[0, 1])` (two independent copies). For a model that fits one T4, TP=1 is fastest.
* OpenAI‑compatible server: `with stokastisk.serving.ServerProcess('Qwen/Qwen2.5-0.5B-Instruct') as srv: srv.wait_ready(); ...` — see the README.
* Unsupported: bfloat16, FlashAttention/FlashInfer/FlashMLA, FP8/FP4, Mistral‑family models.